In [17]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os

In [18]:
load_dotenv()

True

In [19]:
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,    
    api_key=os.getenv("GROQ_API_KEY"),
)

In [20]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [21]:
def create_outline(state: BlogState) -> BlogState:
    title = state['title']
    prompt = f"Generate a detailed outline for the topic: - {title}"
    outline = model.invoke(prompt).content

    state['outline'] = outline

    return state



In [22]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f"Write a detailed prompt on the title: {title} using the following outline: \n {outline}"

    content = model.invoke(prompt).content

    state['content'] = content

    return state



In [23]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [25]:
initial_state = {'title':'Rise of AI Agents'}
final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise of AI Agents', 'outline': '**Outline: “The Rise of AI Agents”**\n\n---\n\n### I. Introduction  \n   A. Definition of AI Agents  \n   1. Software entities that perceive, reason, and act autonomously  \n   2. Distinction between narrow (task‑specific) and general (multi‑domain) agents  \n   3. Key components: sensors, actuators, knowledge base, decision‑making engine  \n   B. Why the topic matters now  \n   1. Convergence of compute, data, and algorithms  \n   2. Economic and societal implications (productivity, labor markets, ethics)  \n   3. Public visibility – chatbots, virtual assistants, autonomous systems  \n\n---\n\n### II. Historical Evolution  \n   A. Early Foundations (1950s‑1970s)  \n   1. Symbolic AI & the “logicist” agent (e.g., SHRDLU)  \n   2. Early robotics and simple reactive agents (Braitenberg vehicles)  \n   B. Knowledge‑Based Systems (1980s)  \n   1. Expert systems (MYCIN, DENDRAL) as rule‑based agents  \n   2. Introduction of the “agent” terminology 